In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-price/train.csv
/kaggle/input/house-price/test.csv


In [2]:
df=pd.read_csv("../input/house-price/train.csv")

In [3]:
df.head(3)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500


In [4]:
cols=df.columns

In [5]:
df.shape

(1460, 81)

In [6]:
# Drop features having more than 70% NAN values
#outliers = [col for col in df.columns if df[col].isna().sum()>=800]

#df=df.drop(outliers,axis=1)

In [7]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [8]:
df=df.drop(['Alley'],axis=1)

In [9]:
df['Id'].isna().sum()

0

**Find Categorical columns**

In [10]:
catCols = [col for col in df.columns if df[col].dtype=="O"]

len(catCols)

42

In [11]:
catCols

['MSZoning',
 'Street',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'PoolQC',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition']

In [12]:
corr=df.corr().abs()

In [13]:
nocat=[col for col in df.columns if df[col].dtype!="O"]
nocat

['Id',
 'MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold',
 'SalePrice']

In [14]:
[col for col in df.columns if df[col].isnull().any()]

['LotFrontage',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

In [15]:
df['LotFrontage']

0       65.0
1       80.0
2       68.0
3       60.0
4       84.0
        ... 
1455    62.0
1456    85.0
1457    66.0
1458    68.0
1459    75.0
Name: LotFrontage, Length: 1460, dtype: float64

In [16]:
df['LotFrontage']=df['LotFrontage'].fillna(0)
meanGBY=np.mean(df['GarageYrBlt'])
df['GarageYrBlt']=df['GarageYrBlt'].fillna(meanGBY)
meanMVA=np.mean(df['MasVnrArea'])
df['MasVnrArea']=df['MasVnrArea'].fillna(meanMVA)

In [17]:
#encode all of the catogorical values
label_encoder = preprocessing.LabelEncoder() 
for i in catCols:
    df[i]= label_encoder.fit_transform(df[i])

In [18]:
import plotly.express as px
fig = px.scatter(df, x="LotArea", y="SalePrice")
fig.show()

In [19]:
fig1 = px.scatter(df, x="OverallQual", y="SalePrice")
fig1.show()

In [20]:
fig2 = px.scatter(df, x="YearBuilt", y="SalePrice")
fig2.show()

In [21]:
fig3 = px.scatter(df, x="TotalBsmtSF", y="SalePrice")
fig3.show()

In [22]:
fig4 = px.histogram(df,x="SalePrice")
fig4.show()

In [23]:
y=df['SalePrice']

In [24]:
y.head(3)

0    208500
1    181500
2    223500
Name: SalePrice, dtype: int64

In [25]:
X=df.drop(['SalePrice'],axis=1)

In [26]:
X.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,3,65.0,8450,1,3,3,0,4,...,0,0,3,4,4,0,2,2008,8,4
1,2,20,3,80.0,9600,1,3,3,0,2,...,0,0,3,4,4,0,5,2007,8,4
2,3,60,3,68.0,11250,1,0,3,0,4,...,0,0,3,4,4,0,9,2008,8,4
3,4,70,3,60.0,9550,1,0,3,0,0,...,0,0,3,4,4,0,2,2006,8,0
4,5,60,3,84.0,14260,1,0,3,0,2,...,0,0,3,4,4,0,12,2008,8,4


**Data Spliting**

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

**Scaling Using Standard Scaler**

In [28]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

**Applying PCA**

In [29]:
from sklearn.decomposition import PCA

pca = PCA()
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

**Training and Prediction Using Linear Regression**

In [30]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(X_train, y_train)
reg.score(X_train, y_train)




0.887296567771538

In [31]:
# Predicting the Test set results
pred = reg.predict(X_test)


In [32]:
# The mean squared error
print("Mean squared error: %.2f" % np.mean((reg.predict(X_test) - y_test) ** 2))
# Explained variance score: 1 is perfect prediction
print('Variance score: %.2f' % reg.score(X_test, y_test))

Mean squared error: 3043210480.16
Variance score: 0.56


**Import Test Data**

In [33]:
dft=pd.read_csv('../input/house-price/test.csv')

In [34]:
dft.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [35]:
#outliers = [col for col in dft.columns if dft[col].isna().sum()>=800]
#dft=dft.drop(outliers,axis=1)


In [36]:
dft=dft.drop(['Alley'],axis=1)

In [37]:
catColst = [col for col in dft.columns if dft[col].dtype=="O"]

In [38]:
dft['LotFrontage']=dft['LotFrontage'].fillna(0)
meanGBY=np.mean(dft['GarageYrBlt'])
dft['GarageYrBlt']=dft['GarageYrBlt'].fillna(meanGBY)
meanMVA=np.mean(dft['MasVnrArea'])
dft['MasVnrArea']=dft['MasVnrArea'].fillna(meanMVA)

In [39]:
#encode all of the catogorical values
label_encoder = preprocessing.LabelEncoder() 
for i in catColst:
    dft[i]= label_encoder.fit_transform(dft[i])

In [40]:
 [col for col in dft.columns if dft[col].dtype=="O"]

[]

In [41]:
nulcol=[col for col in dft.columns if dft[col].isnull().any()]

In [42]:
for i in nulcol:
    dft[i]=dft[i].fillna(0)

In [43]:
dft['BsmtFinSF1']

0        468.0
1        923.0
2        791.0
3        602.0
4        263.0
         ...  
1454       0.0
1455     252.0
1456    1224.0
1457     337.0
1458     758.0
Name: BsmtFinSF1, Length: 1459, dtype: float64

In [44]:

sc = StandardScaler()
test= sc.fit_transform(dft)

In [45]:
pca = PCA()
testdata = pca.fit_transform(test)

In [46]:



result=reg.predict(testdata)
resultData=pd.DataFrame(result,columns=["SalePrice"])
Id=pd.DataFrame(dft["Id"],columns=["Id"])
Result=Id.join(resultData)
Result.to_csv("ResultHousePrices.csv",index=None)